In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import pandas as pd
import yfinance as yf
import utils.data_fetcher as data_fetcher
import utils.data_loader as data_loader
import os
import utils.returns as calc_returns
import core.portfolio as portfolio
import metrics
import risk

Importing utils...
Importing core...
Importing metrics...


In [6]:
# Config
DATA_PATH = 'data/prices.csv'
TICKERS = ['SPY', 'TLT', 'GLD', 'QQQ'] #S&P500, Long term US government bonds, Gold ETF, NASDAQ100

In [7]:
# Caching logic. This could be later on added to /utils
if os.path.exists(DATA_PATH):
    print('Loading cached data...')
    prices = data_loader.load_prices(DATA_PATH)
else:
    print('Fetching new data...')
    prices = data_fetcher.fetch_prices(TICKERS)
    prices.to_csv(DATA_PATH)

Loading cached data...


In [8]:
# !rm data/prices.csv

In [9]:
returns = calc_returns.compute_returns(prices, 'normal')

In [10]:
returns.head()

,GLD,QQQ,SPY,TLT
Date,,,,
2018-01-03,-0.002637,0.009717,0.006325,0.004781
2018-01-04,0.005127,0.001750,0.004215,-0.000159
2018-01-05,-0.001036,0.010043,0.006664,-0.002856
2018-01-08,-0.000160,0.003891,0.001829,-0.000636
2018-01-09,-0.004628,0.000062,0.002263,-0.013373


In [11]:
portfolio_obj = portfolio.Portfolio([0.25, 0.25, 0.25, 0.25], returns)

In [12]:
type(portfolio_obj)

core.portfolio.Portfolio

In [13]:
portfolio_obj.portfolio_returns().head()

Date
2018-01-03    0.004547
2018-01-04    0.002733
2018-01-05    0.003204
2018-01-08    0.001231
2018-01-09   -0.003919
dtype: float64

In [14]:
portfolio_obj.mean_return()

np.float64(0.0004043912652591877)

In [15]:
portfolio_returns = portfolio_obj.portfolio_returns()

In [16]:
type(portfolio_returns)

pandas.core.series.Series

In [17]:
var = metrics.var.historical_var(portfolio_returns)
cvar = metrics.cvar.cvar(portfolio_returns)
drawdown = metrics.drawdown.drawdown(portfolio_returns)

In [18]:
print("var:", var)
print('cvar:', cvar)
print('max drawdown:', drawdown[1])

var: -0.011936897667870249
cvar: -0.01837754859341681
max drawdown: -0.2515453191467012


In [24]:
cov_matrix = risk.covariance.covariance_matrix(returns)

oze


In [ ]:
cov_matrix = f(returns)
cov_matrix

array([[ 7.99886257e-05,  1.50440464e-05,  1.08410077e-05,
         2.81202591e-05],
       [ 1.50440464e-05,  2.48065804e-04,  1.88684817e-04,
        -2.13026334e-05],
       [ 1.08410077e-05,  1.88684817e-04,  1.64959326e-04,
        -2.67177655e-05],
       [ 2.81202591e-05, -2.13026334e-05, -2.67177655e-05,
         1.08558906e-04]])

In [25]:
risk_contribution = risk.risk_contribution.risk_contribution(portfolio_obj.weights, cov_matrix)

In [26]:
type(risk_contribution)

numpy.ndarray

In [28]:
print('Risk Contribution:', risk_contribution)

Risk Contribution: [0.13522283 0.43444017 0.34086512 0.08947188]
